In [1]:
import pandas as pd 
import numpy as np 
import parquet

# maps
import geopandas as gpd
import json
import branca.colormap as cm
import folium
from folium.plugins import TimestampedGeoJson

# visualization
import plotly.graph_objects as go
import seaborn as sns 
import matplotlib.pyplot as plt 



In [ ]:
# add daily rides 
# check numbers for 2016 (too litle trips)
# trip duration in 202 249 missings
# in 24-26 trip_duration=0 included

# где какие сатнции наиболее эффективны, а где избыток. stations capacity meet demand
# Где произошла канибализация, установили больше станций но в общей сложности кол-во райдс не увеличилось
# take the oldest stations and look at their behaviour during 10 years. 

# top-5 stations and top-5 routs. 

# ось времени - первые станции в 5 рафонах: можно таблицой внизу дот мэп

# seasonality 
# users behaviour: Casual/Member


# ML
# can we predict demand?
# can we cluster the stations? 



In [86]:
from pathlib import Path

DATA_DIR = Path('data')


trips = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'start_station_id', 'year'])
trips['year'] = trips['year'].astype(int)  

stations = pd.read_parquet(DATA_DIR / 'stations')[['station_id', 'name', 'neighbourhood', 'ward']]

times = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'start_time', 'end_time', 'trip_duration'])
times.head()




,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [106]:
df = trips.merge(
    stations.rename(columns={
        'station_id': 'start_station_id', 'name': 'start_station_name',
        'neighbourhood': 'start_neighbourhood', 'ward': 'start_ward',
    }),
    on='start_station_id', how='left',
)

In [209]:
df.head()

,trip_id,start_station_id,year,start_station_name,start_neighbourhood,start_ward
0,53279,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
1,53394,7097,2016,Riverdale Park North (Broadview Ave),North Riverdale,Toronto-Danforth
2,58314,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
3,60784,7033,2016,Union Station,St Lawrence-East Bayfront-The Islands,Spadina-Fort York
4,93164,7059,2016,Front St W / Blue Jays Way,Harbourfront-CityPlace,Spadina-Fort York


In [3]:
wards = gpd.read_parquet(DATA_DIR / 'wards' / 'city=toronto' / 'part.parquet')

In [6]:
# print(wards[['ward']].sort_values('ward'))
# print(wards.crs)

# Some Vanity metrics (absolute values)

## Stations

In [210]:
stations_yearly = trips.groupby('year')['start_station_id'].nunique().reset_index(name='nr_stations')

In [320]:
stations_yearly = stations_yearly.assign(
    yoy_change=lambda d: d['nr_stations'].diff(),
    yoy_pct_change=lambda d: d['nr_stations'].pct_change(),
)

stations_count_labels = [f"{v}" for v in stations_yearly['nr_stations']]
stations_pct_labels = [
    '' if pd.isna(p) else f"{p:+.0%}"
    for p in stations_yearly['yoy_pct_change']
]
# below 10% YoY growth reads as a slowdown (navy); 10%+ as healthy growth (orange) -
# warm/cool rather than red/green, which is one of the least colorblind-safe
# pairings
stations_pct_colors = [
    '#888888' if pd.isna(p) else ('#1B3A5C' if p < 0.10 else '#FF6B4A')
    for p in stations_yearly['yoy_pct_change']
]
yoy_hover = [
    '' if pd.isna(v) else f"<br><b>YoY change:</b> {v:+.0f}"
    for v in stations_yearly['yoy_change']
]

fig_stations_per_year = go.Figure()

fig_stations_per_year.add_trace(go.Scatter(
    x=stations_yearly['year'],
    y=stations_yearly['nr_stations'],
    mode='lines+markers+text',
    text=stations_count_labels,
    textposition='top center',
    cliponaxis=False,
    line=dict(color="#FF6B4A", width=3),
    marker=dict(color='#FF6B4A', size=10, line=dict(color='white', width=2)),
    customdata=yoy_hover,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}%{customdata}<extra></extra>',
))

pct_offset = (stations_yearly['nr_stations'].max() - stations_yearly['nr_stations'].min()) * 0.09

fig_stations_per_year.add_trace(go.Scatter(
    x=stations_yearly['year'],
    y=stations_yearly['nr_stations'] - pct_offset,
    mode='text',
    text=stations_pct_labels,
    textposition='bottom center',
    cliponaxis=False,
    textfont=dict(color=stations_pct_colors, size=11),
    hoverinfo='skip',
    showlegend=False,
))

fig_stations_per_year.update_layout(
    title=dict(text='Stations Per Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black',tickvals=stations_yearly['year']),
    yaxis=dict(title='Stations', showgrid=False, showline=True, linecolor='black', range=[0, stations_yearly['nr_stations'].max() * 1.18]),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    margin=dict(t=100),
)

fig_stations_per_year

## Trips

In [ ]:
# There have been more than 5.3 million trips so far in 2024 (data from January to September)
# That's an average of 19,500 trips per day.
# That's 50% more trips per day than just 2022, and almost double the average in 2021.

In [212]:
trips_yearly = (trips
                    .groupby('year')['start_station_id']
                    .size()
                    .reset_index(name='nr_trips')
                    )

In [213]:
trips_yearly.loc[trips_yearly['year']==2016, 'nr_trips'] = 830000

In [321]:
trips_main = trips_yearly[trips_yearly['year'] <= 2025]
trips_connector = trips_yearly[trips_yearly['year'].isin([2025, 2026])]
trips_last_point = trips_yearly[trips_yearly['year'] == 2026]

fig_trips_per_year = go.Figure()

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_main['year'],
    y=trips_main['nr_trips'],
    mode='lines+markers+text',
    text=[f"{v/1e6:.1f}M" for v in trips_main['nr_trips']],
    textposition='top center',
    line=dict(color="#192562", width=3),
    marker=dict(color='#192562', size=10, line=dict(color='white', width=2)),
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<extra></extra>',
))

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_connector['year'],
    y=trips_connector['nr_trips'],
    mode='lines',
    line=dict(color='#B0B5BA', width=3, dash='dash'),
    showlegend=False,
    hoverinfo='skip',
))

fig_trips_per_year.add_trace(go.Scatter(
    x=trips_last_point['year'],
    y=trips_last_point['nr_trips'],
    mode='markers+text',
    text=[f"{v/1e6:.1f}M" for v in trips_last_point['nr_trips']],
    textposition='top center',
    marker=dict(color='#B0B5BA', size=10, line=dict(color='white', width=2)),
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<extra></extra>',
    showlegend=False,
))

fig_trips_per_year.update_layout(
    title=dict(text='Trips Per Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black',tickvals=stations_yearly['year']),
    yaxis=dict(title='Trips', showgrid=False, showline=True, linecolor='black',  dtick=1000000, ticks='outside', ticklen=6, tickcolor='black'),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    
    autosize=True,
    height=550,
)

fig_trips_per_year


## Users

In [215]:
users = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'user_type'])

In [216]:
trips_usertype = (users
    .groupby(['year', 'user_type'])['trip_id']
    .size()
    .reset_index(name = 'nr_trips')
    )

In [217]:
trips_usertype.head() 

,year,user_type,nr_trips
0,2016,Casual,122484
1,2016,Member,245477
2,2017,Casual,327585
3,2017,Member,1164782
4,2018,Annual Member,1572980


# Some ratios 

In [166]:
trips.head()

,trip_id,start_station_id,year
0,53279,7100,2016
1,53394,7097,2016
2,58314,7100,2016
3,60784,7033,2016
4,93164,7059,2016


### trips per day

In [169]:
trips_per_day = (trips
    .groupby('year')
    .size()
    .reset_index(name='trips')
    .assign(trips_per_day = lambda d: round(d['trips']/365))
    )

trips_per_day

,year,trips,trips_per_day
0,2016,367961,1008.0
1,2017,1492367,4089.0
2,2018,1922955,5268.0
3,2019,2439517,6684.0
4,2020,2911308,7976.0
5,2021,3575182,9795.0
6,2022,4620469,12659.0
7,2023,5713141,15652.0
8,2024,6953094,19050.0
9,2025,7812520,21404.0


In [158]:
times.head()

,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [ ]:
times

In [ ]:
times.start_time.

0            9
1            9
2           10
3           11
4           18
            ..
44638683    30
44638684    30
44638685    30
44638686    30
44638687    30
Name: start_time, Length: 44638688, dtype: int32

### trips per station

In [218]:
trips_per_station = (trips
    .groupby('year', as_index = False)
    .agg(stations = ('start_station_id', 'nunique'),
         trips = ('start_station_id', 'size')))

trips_per_station.loc[trips_per_station['year']==2016, 'trips'] = 830000

trips_per_station = (trips_per_station
    .assign(trips_per_station = lambda d: round(d['trips']/d['stations']))
    )



In [219]:
trips_per_station = (trips_per_station
    .assign(yoy_fleet_change = lambda d: d['stations'].diff(),
            yoy_fleet_pct_change = lambda d: d['stations'].pct_change().round(3),
            yoy_trips_change = lambda d: d['trips'].diff(),
            yoy_pct_trips_change = lambda d: d['trips'].pct_change().round(3),
            yoy_tps_change = lambda d: d['trips_per_station'].diff(),
            yoy_pct_tps_change = lambda d: d['trips_per_station'].pct_change().round(3))
    )
trips_per_station

,year,stations,trips,trips_per_station,yoy_fleet_change,yoy_fleet_pct_change,yoy_trips_change,yoy_pct_trips_change,yoy_tps_change,yoy_pct_tps_change
0,2016,200,830000,4150.0,NaN,NaN,<NA>,<NA>,<NA>,<NA>
1,2017,265,1492367,5632.0,65.0,0.325,662367,0.798,1482.0,0.357
2,2018,359,1922955,5356.0,94.0,0.355,430588,0.289,-276.0,-0.049
3,2019,469,2439517,5202.0,110.0,0.306,516562,0.269,-154.0,-0.029
4,2020,611,2911308,4765.0,142.0,0.303,471791,0.193,-437.0,-0.084
5,2021,627,3575182,5702.0,16.0,0.026,663874,0.228,937.0,0.197
6,2022,682,4620469,6775.0,55.0,0.088,1045287,0.292,1073.0,0.188
7,2023,814,5713141,7019.0,132.0,0.194,1092672,0.236,244.0,0.036
8,2024,879,6953094,7910.0,65.0,0.080,1239953,0.217,891.0,0.127
9,2025,1048,7812520,7455.0,169.0,0.192,859426,0.124,-455.0,-0.058


In [324]:
from plotly.subplots import make_subplots

# orange = expansion year (fleet grew 10%+, same threshold as the Stations panel)
# that likely diluted demand; navy = a slower/absorption year where trips per
# station had room to recover. warm/cool rather than red/green, which is one of
# the least colorblind-safe pairings
bar_colors = [
    '#B0B5BA' if y in (2016, 2026) else ('#FF6B4A' if p >= 0.10 else '#1B3A5C')
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]

cycle_stage = [
    'Incomplete year' if y in (2016, 2026) else (
        'Expansion — fleet growing fast' if p >= 0.10 else 'Absorption — demand catching up'
    )
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]
bar_opacities = [
    1.0 if y == 2024 else (0.6 if y in (2016, 2026) else 0.45)
    for y in trips_per_station['year']
]

fig_trips_per_station = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.65, 0.35],
)

fig_trips_per_station.add_trace(go.Bar(
    x=trips_per_station['year'],
    y=trips_per_station['trips_per_station'],
    marker=dict(color=bar_colors, opacity=bar_opacities),
    name='Trips per station',
    customdata=cycle_stage,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<br><b>Stage:</b> %{customdata}<extra></extra>',
), row=1, col=1)

tps_line = trips_per_station[trips_per_station['year'] <= 2025]
connector = trips_per_station[trips_per_station['year'].isin([2025, 2026])]
last_point = trips_per_station[trips_per_station['year'] == 2026]

fig_trips_per_station.add_trace(go.Scatter(
    x=tps_line['year'],
    y=tps_line['yoy_fleet_pct_change'] * 100,
    mode='lines+markers',
    line=dict(color="#040404", width=3),
    marker=dict(color="#040404", size=8, line=dict(color='white', width=2)),
    name='YoY % fleet change',
    hovertemplate='<b>Year:</b> %{x}<br><b>Percentage:</b> %{y:.1f}%<extra></extra>',
), row=2, col=1)

fig_trips_per_station.add_trace(go.Scatter(
    x=connector['year'],
    y=connector['yoy_fleet_pct_change'] * 100,
    mode='lines',
    line=dict(color='#B0B5BA', width=3, dash='dash'),
    showlegend=False,
    hoverinfo='skip',
), row=2, col=1)

fig_trips_per_station.add_trace(go.Scatter(
    x=last_point['year'],
    y=last_point['yoy_fleet_pct_change'] * 100,
    mode='markers',
    marker=dict(color='#B0B5BA', size=8, line=dict(color='white', width=2)),
    showlegend=False,
    hovertemplate='<b>Year:</b> %{x}<br><b>Percentage:</b> %{y:.1f}%<extra></extra>',
), row=2, col=1)

pct_max = (trips_per_station['yoy_fleet_pct_change'] * 100).max()

fig_trips_per_station.update_layout(
    title=dict(text='Trips Per Station, by Year', x=0.5),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
)

fig_trips_per_station.update_xaxes(showgrid=False, showline=True, linecolor='black', tickmode='array', tickvals=trips_per_station['year'], row=1, col=1)
fig_trips_per_station.update_xaxes(title='Year', showgrid=False, showline=True, linecolor='black', tickmode='array', tickvals=trips_per_station['year'], row=2, col=1)
fig_trips_per_station.update_yaxes(title='Trips per station', showgrid=False, showline=True, linecolor='black', row=1, col=1)
fig_trips_per_station.update_yaxes(title='YoY % fleet change', showgrid=False, showline=True, linecolor='black', ticksuffix='%', range=[0, pct_max * 1.15], row=2, col=1)

fig_trips_per_station.write_html(
    'trips_per_station_chart.html',
    include_plotlyjs='cdn',
    full_html=True,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

fig_trips_per_station

In [325]:
fig_stacked = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.45, 0.55],
    subplot_titles=('Stations', 'Trips per Station'),
)

stations_count_labels = [f"{v}" for v in trips_per_station['stations']]
stations_pct_labels = [
    '' if pd.isna(p) else f"{p:+.0%}"
    for p in trips_per_station['yoy_fleet_pct_change']
]
# below 10% YoY growth reads as a slowdown (navy); 10%+ as healthy growth (orange) -
# warm/cool rather than red/green, which is one of the least colorblind-safe
# pairings
stations_pct_colors = [
    '#888888' if pd.isna(p) else ('#1B3A5C' if p < 0.10 else '#FF6B4A')
    for p in trips_per_station['yoy_fleet_pct_change']
]

yoy_hover = [
    '' if pd.isna(v) else f"<br><b>YoY change:</b> {v:+.0f}"
    for v in trips_per_station['yoy_fleet_change']
]

fig_stacked.add_trace(go.Scatter(
    x=trips_per_station['year'],
    y=trips_per_station['stations'],
    mode='lines+markers+text',
    text=stations_count_labels,
    textposition='top center',
    cliponaxis=False,
    line=dict(color='#FF6B4A', width=3),
    marker=dict(color='#FF6B4A', size=10, line=dict(color='white', width=2)),
    customdata=yoy_hover,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}%{customdata}<extra></extra>',
), row=1, col=1)

pct_offset = (trips_per_station['stations'].max() - trips_per_station['stations'].min()) * 0.09

fig_stacked.add_trace(go.Scatter(
    x=trips_per_station['year'],
    y=trips_per_station['stations'] - pct_offset,
    mode='text',
    text=stations_pct_labels,
    textposition='bottom center',
    cliponaxis=False,
    textfont=dict(color=stations_pct_colors, size=11),
    hoverinfo='skip',
    showlegend=False,
), row=1, col=1)

# orange = expansion year (fleet grew 10%+, same threshold as the Stations panel)
# that likely diluted demand; navy = a slower/absorption year where trips per
# station had room to recover. warm/cool rather than red/green, which is one of
# the least colorblind-safe pairings
bar_colors = [
    '#B0B5BA' if y in (2016, 2026) else ('#FF6B4A' if p >= 0.10 else '#1B3A5C')
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]

cycle_stage = [
    'Incomplete year' if y in (2016, 2026) else (
        'Expansion — fleet growing fast' if p >= 0.10 else 'Absorption — demand catching up'
    )
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]
bar_opacities = [1.0 if y == 2024 else (0.6 if y in (2016, 2026) else 0.45) for y in trips_per_station['year']]

fig_stacked.add_trace(go.Bar(
    x=trips_per_station['year'],
    y=trips_per_station['trips_per_station'],
    marker=dict(color=bar_colors, opacity=bar_opacities),
    customdata=cycle_stage,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<br><b>Stage:</b> %{customdata}<extra></extra>',
), row=2, col=1)

fig_stacked.update_layout(
    # title=dict(text='Stations and Trips per Station, by Year', x=0.5),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
    height=800,
    margin=dict(t=100),
)

fig_stacked.update_xaxes(showgrid=False, showline=True, linecolor='black', tickmode='array', tickvals=trips_per_station['year'], row=1, col=1)
fig_stacked.update_xaxes(title='Year', showgrid=False, showline=True, linecolor='black', tickmode='array', tickvals=trips_per_station['year'], row=2, col=1)
fig_stacked.update_yaxes(title='Stations', showgrid=False, showline=True, linecolor='black', range=[0, trips_per_station['stations'].max() * 1.18], row=1, col=1)
fig_stacked.update_yaxes(title='Trips per station', showgrid=False, showline=True, linecolor='black', row=2, col=1)

fig_stacked.write_html(
    'stations_trips_per_station_chart.html',
    include_plotlyjs='cdn',
    full_html=True,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

fig_stacked

In [ ]:
# Station rollout came in bursts, creating a repeating expand-and-absorb cycle. 
# Fast growth from 2017 to 2020 spread ridership thin, so trips per station fell. 
# Slower expansion in 2021–2022 let demand catch up. 
# Trips per station peaked at 7910 in 2024 as the network neared capacity. 
# A 19.2% fleet expansion in 2025 then eased the pressure. 2026 is excluded because its data is incomplete.

### average/median trip duration

In [148]:
times.head()

,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [149]:
times.isna().groupby(times['year']).sum()

,trip_id,year,start_time,end_time,trip_duration
year,,,,,
2016,0,0,0,0,0
2017,0,0,0,1,0
2018,0,0,0,0,0
2019,0,0,0,0,16
2020,0,0,0,0,249
2021,0,0,0,0,0
2022,0,0,0,0,0
2023,0,0,0,0,0
2024,0,0,0,5822,0


In [ ]:
times[(times['trip_duration']==0) & times['end_time'].isna()].groupby('year').size() # still junk data issue

year
2017       1
2024    5822
2025    1092
2026    8065
dtype: int64

In [20]:
mask = (times['trip_duration']==0) & times['end_time'].isna()
times = times[~mask]

In [156]:
times.isna().groupby(times['year']).sum()

,trip_id,year,start_time,end_time,trip_duration
year,,,,,
2016,0,0,0,0,0
2017,0,0,0,0,0
2018,0,0,0,0,0
2019,0,0,0,0,16
2020,0,0,0,0,249
2021,0,0,0,0,0
2022,0,0,0,0,0
2023,0,0,0,0,0
2024,0,0,0,0,0


In [21]:
mask = times['trip_duration'].isna() & ((times['end_time']-times['start_time']).dt.total_seconds() <= 60)
times = times[~mask]

In [22]:
times.head()

,trip_id,year,start_time,end_time,trip_duration
0,53279,2016,2016-07-09 01:03:00,2016-07-09 01:15:00,714
1,53394,2016,2016-07-09 02:15:00,2016-07-09 02:22:00,417
2,58314,2016,2016-07-10 17:04:00,2016-07-10 17:36:00,1904
3,60784,2016,2016-07-11 01:45:00,2016-07-11 01:58:00,784
4,93164,2016,2016-07-18 13:35:00,2016-07-18 13:42:00,443


In [26]:
times.dtypes

trip_id                   Int64
year                   category
start_time       datetime64[us]
end_time         datetime64[us]
trip_duration             Int32
dtype: object

In [30]:
mask = times['trip_duration'].isna()
times.loc[mask, 'trip_duration'] = (times[mask]['end_time']-times[mask]['start_time']).dt.total_seconds().astype('int32')

In [36]:
times = times[times['trip_duration']!=0]

In [41]:
trip_duration_stats = (times
    .groupby('year', as_index=False)
    .agg(mean_duration = ('trip_duration', 'mean'),
         median_duration = ('trip_duration', 'median'),
         std_duration = ('trip_duration', 'std')
    ))
duration_cols = ['mean_duration', 'median_duration', 'std_duration']
trip_duration_stats[[f'{c}_sec' for c in duration_cols]] = trip_duration_stats[duration_cols].round(0)
trip_duration_stats[[f'{c}_min' for c in duration_cols]] = (trip_duration_stats[duration_cols]/60).round(0)

In [48]:
trip_duration_stats[['year'] + [f'{c}_min' for c in duration_cols]]

,year,mean_duration_min,median_duration_min,std_duration_min
0,2016,18.0,12.0,23.0
1,2017,17.0,11.0,163.0
2,2018,16.0,11.0,27.0
3,2019,17.0,11.0,309.0
4,2020,20.0,14.0,127.0
5,2021,17.0,13.0,73.0
6,2022,16.0,12.0,75.0
7,2023,18.0,11.0,147.0
8,2024,16.0,11.0,90.0
9,2025,16.0,11.0,24.0


In [ ]:
# median duration is quite steady

## Regressions: 

### Delta trips~ delta stations (panel + first differeciation)
### Poisson trips ~ workday + month + hour
### trip_duration ~ workday + month + hour


In [85]:
df.head()

,start_station_id,year,start_station_name,start_neighbourhood,start_ward
0,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
1,7097,2016,Riverdale Park North (Broadview Ave),North Riverdale,Toronto-Danforth
2,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
3,7033,2016,Union Station,St Lawrence-East Bayfront-The Islands,Spadina-Fort York
4,7059,2016,Front St W / Blue Jays Way,Harbourfront-CityPlace,Spadina-Fort York


In [95]:
df_regr = df[~df.year.isin([2016, 2026])]

In [96]:
panel = (df_regr
            .groupby(['year', 'start_ward'], as_index=False)
            .agg(stations = ('start_station_id', 'nunique'),
                trips = ('trip_id', 'size'))
            .sort_values(['start_ward', 'year']) 
            
            .assign(stations_lag=lambda d: d.groupby('start_ward')['stations'].shift(1),
                    trips_lag=lambda d: d.groupby('start_ward')['trips'].shift(1))
            
            .assign(
                delta_stations=lambda d: d['stations'] - d['stations_lag'],
                delta_trips=lambda d: d['trips'] - d['trips_lag']
            )
            .dropna(subset=['delta_stations', 'delta_trips'])
 )

In [97]:
panel

,year,start_ward,stations,trips,stations_lag,trips_lag,delta_stations,delta_trips
10,2018,Beaches-East York,16,12207,7.0,4634,9.0,7573
22,2019,Beaches-East York,32,36629,16.0,12207,16.0,24422
35,2020,Beaches-East York,42,74439,32.0,36629,10.0,37810
54,2021,Beaches-East York,42,81007,42.0,74439,0.0,6568
73,2022,Beaches-East York,49,87621,42.0,81007,7.0,6614
...,...,...,...,...,...,...,...,...
72,2021,York South-Weston,4,4738,4.0,3419,0.0,1319
91,2022,York South-Weston,4,5179,4.0,4738,0.0,441
112,2023,York South-Weston,23,7331,4.0,5179,19.0,2152
135,2024,York South-Weston,27,15241,23.0,7331,4.0,7910


In [98]:
import statsmodels.api as sm

X = sm.add_constant(panel['delta_stations'])
y = panel['delta_trips']



In [101]:
model = sm.OLS(y, X).fit()

In [102]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            delta_trips   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     10.06
Date:                Thu, 05 Nov 2026   Prob (F-statistic):            0.00187
Time:                        16:15:46   Log-Likelihood:                -1748.5
No. Observations:                 136   AIC:                             3501.
Df Residuals:                     134   BIC:                             3507.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const           2.356e+04   1.08e+04      2.

In [151]:
model_cl = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': panel['start_ward']})
print(model_cl.summary())

                            OLS Regression Results                            
Dep. Variable:            delta_trips   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     1.797
Date:                Fri, 06 Nov 2026   Prob (F-statistic):              0.194
Time:                        00:23:09   Log-Likelihood:                -1748.5
No. Observations:                 136   AIC:                             3501.
Df Residuals:                     134   BIC:                             3507.
Df Model:                           1                                         
Covariance Type:              cluster                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           2.356e+04   1.45e+04      1.

## MAPS


### WARD MAP

In [234]:
years = sorted(df['year'].unique())

In [311]:
ward_year_raw = {}
ward_year_log = {}
all_log_values = []

for year in years:
    trips_year = df[df['year'] == year]
    counts = trips_year.groupby('start_ward').size()
    ward_year_raw[year] = counts
    log_counts = np.log(counts)
    ward_year_log[year] = log_counts
    all_log_values.extend(log_counts.values)

vmin, vmax = min(all_log_values), max(all_log_values)
colormap = cm.linear.YlGn_09.scale(vmin, vmax)

features = []
for ward_name, geom in zip(wards['ward'], wards['geometry']):
    geom_json = geom.__geo_interface__
    for year in years:
        log_count = ward_year_log[year].get(ward_name, vmin)
        raw_count = int(ward_year_raw[year].get(ward_name, 0))
        color = colormap(log_count)
        features.append({
            "type": "Feature",
            "geometry": geom_json,
            "properties": {
                "times": [f"{year}-01-01"],
                "style": {
                    "color": "white",
                    "fillColor": color,
                    "fillOpacity": 0.85,
                    "weight": 1
                },
                "tooltip": f"{ward_name}: {raw_count:,} trips ({year})"
            }
        })

geo_timed = {"type": "FeatureCollection", "features": features}
print(len(features), "features built")

275 features built


In [ ]:
m = folium.Map(
    location=[43.72, -79.42],
    zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=False,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)

TimestampedGeoJson(
    geo_timed,
    period='P1Y',
    duration='P1Y',
    auto_play=False,
    loop=False,
    date_options='YYYY',
    transition_time=300,
).add_to(m)

colormap.caption = 'Trips started (log scale)'
colormap.add_to(m)

m.save('ward_map_timelapse.html')
print("saved ward_map_timelapse.html")

### WARD MAP, 2026 trips (static snapshot, variant — does not replace the timelapse above)

In [286]:
# official Toronto 2018 ward numbering
WARD_NUMBERS = {
    'Etobicoke North': 1, 'Etobicoke Centre': 2, 'Etobicoke-Lakeshore': 3,
    'Parkdale-High Park': 4, 'York South-Weston': 5, 'York Centre': 6,
    'Humber River-Black Creek': 7, 'Eglinton-Lawrence': 8, 'Davenport': 9,
    'Spadina-Fort York': 10, 'University-Rosedale': 11, "Toronto-St. Paul's": 12,
    'Toronto Centre': 13, 'Toronto-Danforth': 14, 'Don Valley West': 15,
    'Don Valley East': 16, 'Don Valley North': 17, 'Willowdale': 18,
    'Beaches-East York': 19, 'Scarborough Southwest': 20, 'Scarborough Centre': 21,
    'Scarborough-Agincourt': 22, 'Scarborough North': 23, 'Scarborough-Guildwood': 24,
    'Scarborough-Rouge Park': 25,
}

df_2026 = df[df['year'] == 2026]

trips_by_ward_2026 = (
    df_2026.groupby('start_ward').size()
    .reindex(wards['ward']).fillna(0).astype(int)
)
stations_by_ward_2026 = (
    df_2026.groupby('start_ward')['start_station_id'].nunique()
    .reindex(wards['ward']).fillna(0).astype(int)
)
# raw trip counts are dominated by downtown (~6,450x spread between busiest and
# quietest ward) purely because downtown has far more stations, not because each
# station there is used that much harder -> normalize by station count per ward.
# that narrows the spread to ~170x; still wide, so color (not the displayed count)
# is additionally log-scaled for legibility, same approach as the ward timelapse map.
trips_per_station_2026 = (trips_by_ward_2026 / stations_by_ward_2026.replace(0, pd.NA)).round(1)

# simplify boundary detail for a cleaner, blockier look (matches the reference map's style)
wards_simple = wards.copy()
wards_simple['geometry'] = wards_simple.geometry.simplify(tolerance=0.0012, preserve_topology=True)
wards_simple['centroid'] = wards_simple.geometry.centroid

log_tps = np.log(trips_per_station_2026.astype(float))
colormap_wards_2026 = cm.LinearColormap(
    colors=['#EDD9F0', '#9B59B6', '#4A235A'],
    vmin=log_tps.min(),
    vmax=log_tps.max(),
)
colormap_wards_2026.caption = 'Trips per station, 2026 (log scale)'

print(trips_by_ward_2026.to_frame('trips').join(stations_by_ward_2026.rename('stations')).join(trips_per_station_2026.rename('trips_per_station')).sort_values('trips', ascending=False))

                            trips  stations  trips_per_station
ward                                                          
Spadina-Fort York         2727692       187            14586.6
University-Rosedale       1337004       120            11141.7
Toronto Centre            1314137       103            12758.6
Toronto-Danforth           394369        76             5189.1
Davenport                  313052        64             4891.4
Parkdale-High Park         287015        68             4220.8
Toronto-St. Paul's         129342        60             2155.7
Beaches-East York          106150        52             2041.3
Etobicoke-Lakeshore         87066        38             2291.2
Don Valley West             30074        32              939.8
Eglinton-Lawrence           19231        27              712.3
York South-Weston           15876        34              466.9
Scarborough Southwest       15342        39              393.4
Willowdale                  10895        14            

/var/folders/xf/ygl5_zk57kn1rh2h1xj9fyx40000gn/T/ipykernel_1611/769431856.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  wards_simple['centroid'] = wards_simple.geometry.centroid


In [ ]:
wards_simple['number'] = wards_simple['ward'].map(WARD_NUMBERS)
wards_simple['trips_2026'] = wards_simple['ward'].map(trips_by_ward_2026)
wards_simple['stations_2026'] = wards_simple['ward'].map(stations_by_ward_2026)
wards_simple['trips_per_station_2026'] = wards_simple['ward'].map(trips_per_station_2026)
wards_simple['color_value'] = wards_simple['ward'].map(log_tps)


def _tooltip_html(row):
    return (
        f'<div style="font-weight:700; font-size:15px; margin-bottom:6px;">'
        f'{row["ward"]}</div>'
        '<table style="border-collapse:collapse;">'
        f'<tr><td style="font-weight:700; padding-right:16px;">Rides</td>'
        f'<td>{row["trips_2026"]:,}</td></tr>'
        f'<tr><td style="font-weight:700; padding-right:16px;">Stations</td>'
        f'<td>{row["stations_2026"]:,}</td></tr>'
        f'<tr><td style="font-weight:700; padding-right:16px;">Rides per Station</td>'
        f'<td>{row["trips_per_station_2026"]:,.1f}</td></tr>'
        '</table>'
    )

wards_simple['tooltip_html'] = wards_simple.apply(_tooltip_html, axis=1)

wards_geojson = wards_simple[
    ['ward', 'trips_2026', 'stations_2026', 'trips_per_station_2026', 'color_value',
     'tooltip_html', 'geometry']
].__geo_interface__

m_wards_2026 = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=False,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)

folium.GeoJson(
    wards_geojson,
    style_function=lambda feature: {
        'fillColor': colormap_wards_2026(feature['properties']['color_value']),
        'fillOpacity': 0.85,
        'color': 'white',
        'weight': 1.5,
    },
    highlight_function=lambda feature: {'weight': 3, 'color': '#1B3A5C'},
    tooltip=folium.GeoJsonTooltip(
        fields=['tooltip_html'],
        labels=False,
        sticky=True,
        style=(
            'background-color: white; border: none; border-radius: 10px; '
            'box-shadow: 0 2px 10px rgba(0,0,0,0.25); padding: 12px 16px; '
            "font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; "
            'font-size: 14px; color: #1a1a1a;'
        ),
    ),
).add_to(m_wards_2026)

# cast numpy float64 -> native float, otherwise fit_bounds' JSON serialization fails
bounds_wards = [float(v) for v in wards_simple.total_bounds]  # [minx, miny, maxx, maxy]
m_wards_2026.fit_bounds([[bounds_wards[1], bounds_wards[0]], [bounds_wards[3], bounds_wards[2]]])

colormap_wards_2026.add_to(m_wards_2026)

m_wards_2026.save('ward_map_2026.html')
print("saved ward_map_2026.html")

saved ward_map_2026.html


### combined map with dots and wards

In [308]:

station_trips_2026 = df_2026.groupby('start_station_id').size().rename('trips_2026')

stations_2026_pts = (
    stations_full.set_index('station_id')
    .join(station_trips_2026, how='inner')
    .dropna(subset=['lat', 'lon'])
    .reset_index()
)

scaled = np.sqrt(stations_2026_pts['trips_2026'])
scaled_norm = (scaled - scaled.min()) / (scaled.max() - scaled.min())
stations_2026_pts['radius_px'] = MIN_RADIUS_PX + scaled_norm * (MAX_RADIUS_PX - MIN_RADIUS_PX)

m_combined = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=True,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)

# ward layer first (bottom), lighter fill so the station dots stay visible on top of it
folium.GeoJson(
    wards_geojson,
    style_function=lambda feature: {
        'fillOpacity': 0,
        'color': '#1B3A5C',
        'weight': 1.5,
    },
    highlight_function=lambda feature: {'weight': 3, 'color': '#1B3A5C', 'fillColor': '#1B3A5C', 'fillOpacity': 0.15},
    tooltip=folium.GeoJsonTooltip(
        fields=['tooltip_html'],
        labels=False,
        sticky=True,
        style=(
            'background-color: white; border: none; border-radius: 10px; '
            'box-shadow: 0 2px 10px rgba(0,0,0,0.25); padding: 12px 16px; '
            "font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; "
            'font-size: 14px; color: #1a1a1a;'
        ),
    ),
).add_to(m_combined)

# station dots on top
for row in stations_2026_pts.itertuples():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=row.radius_px,
        fill=True,
        fill_color=DOT_COLOR,
        fill_opacity=0.85,
        stroke=True,
        color='white',
        weight=1,
        tooltip=f"{row.name}: {row.trips_2026:,} rides (2026)",
    ).add_to(m_combined)

bounds_wards = [float(v) for v in wards_simple.total_bounds]
m_combined.fit_bounds([[bounds_wards[1], bounds_wards[0]], [bounds_wards[3], bounds_wards[2]]])

m_combined.save('ward_and_stations_map_2026.html')
print("saved ward_and_stations_map_2026.html")

saved ward_and_stations_map_2026.html


###  STATIONS MAP (dot timelapse)

In [237]:
stations_full = pd.read_parquet(DATA_DIR / 'stations')[['station_id', 'name', 'lat', 'lon']]
first_year = trips.groupby('start_station_id')['year'].min().rename('first_year')
total_trips = trips.groupby('start_station_id').size().rename('total_trips')


stations_tl = (
    stations_full.set_index('station_id')
    .join(first_year, how='inner')
    .join(total_trips, how='inner')
    .dropna(subset=['lat', 'lon'])
    .reset_index()
)
print(stations_tl.shape, '/', stations_full.shape[0], 'stations plotted')
print(stations_tl['first_year'].value_counts().sort_index())

(1192, 6) / 1202 stations plotted
first_year
2016    200
2017     65
2018     93
2019    107
2020    146
2021     18
2022     66
2023    151
2024     88
2025    177
2026     81
Name: count, dtype: int64


### dot map - old version

In [291]:

MIN_RADIUS_PX, MAX_RADIUS_PX = 3, 12
DOT_COLOR = "#FF6B4A"  
                        

scaled = np.sqrt(stations_tl['total_trips'])
scaled_norm = (scaled - scaled.min()) / (scaled.max() - scaled.min())
stations_tl['radius_px'] = MIN_RADIUS_PX + scaled_norm * (MAX_RADIUS_PX - MIN_RADIUS_PX)

station_features = []
for row in stations_tl.itertuples():
    station_features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(row.lon), float(row.lat)]},
        "properties": {

            "times": [f"{row.first_year}-01-01T12:00:00"],
            "icon": "circle",
            "iconstyle": {
                "fillColor": DOT_COLOR, "fillOpacity": 0.7,
                "stroke": True, "color": "white", "weight": 1,
                "radius": float(row.radius_px),  
            },
            "tooltip": row.name,
        },
    })


last_year = int(stations_tl['first_year'].max())
station_features.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [float(stations_tl['lon'].iloc[0]), float(stations_tl['lat'].iloc[0])]},
    "properties": {
        "times": [f"{last_year}-12-31T12:00:00"],
        "icon": "circle",
        "iconstyle": {"radius": 0, "fillOpacity": 0, "opacity": 0},
    },
})

geo_stations_timed = {"type": "FeatureCollection", "features": station_features}
print(len(station_features), "station features built (incl. 1 padding dummy)")

1193 station features built (incl. 1 padding dummy)


In [292]:
bounds = [
    [float(stations_tl['lat'].min()), float(stations_tl['lon'].min())],
    [float(stations_tl['lat'].max()), float(stations_tl['lon'].max())],
]

m = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=False,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)
m.fit_bounds(bounds)

TimestampedGeoJson(
    geo_stations_timed,
    period='P1Y',
    duration=None, 
    auto_play=True,
    loop=True,
    date_options='YYYY',
    transition_time=300,
).add_to(m)

m.save('stations_map_timelapse.html')
print("saved stations_map_timelapse.html")

saved stations_map_timelapse.html


### STATIONS MAP colored by first year - actual version in overview

In [293]:
# on-brand sequential ramp (light -> dark orange), one hue, mapped to first_year
colormap_years = cm.LinearColormap(
    colors=['#FFE0D6', '#FF6B4A', '#8B2E1A'],
    vmin=stations_tl['first_year'].min(),
    vmax=stations_tl['first_year'].max(),
)
colormap_years.caption = 'First year'

station_features_colored = []
for row in stations_tl.itertuples():
    station_features_colored.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(row.lon), float(row.lat)]},
        "properties": {
            "times": [f"{row.first_year}-01-01T12:00:00"],
            "icon": "circle",
            "iconstyle": {
                "fillColor": colormap_years(row.first_year), "fillOpacity": 0.8,
                "stroke": True, "color": "white", "weight": 1,
                "radius": float(row.radius_px),
            },
            "tooltip": f"{row.name} ({row.first_year})",
        },
    })

station_features_colored.append({
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [float(stations_tl['lon'].iloc[0]), float(stations_tl['lat'].iloc[0])]},
    "properties": {
        "times": [f"{last_year}-12-31T12:00:00"],
        "icon": "circle",
        "iconstyle": {"radius": 0, "fillOpacity": 0, "opacity": 0},
    },
})

geo_stations_timed_colored = {"type": "FeatureCollection", "features": station_features_colored}
print(len(station_features_colored), "colored station features built")

1193 colored station features built


In [294]:
m_colored = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ',
    zoom_control=False,
    scrollWheelZoom=False,
    doubleClickZoom=False,
    touchZoom=False,
    boxZoom=False,
    keyboard=False,
)
m_colored.fit_bounds(bounds)

TimestampedGeoJson(
    geo_stations_timed_colored,
    period='P1Y',
    duration=None,
    auto_play=True,
    loop=True,
    date_options='YYYY',
    transition_time=600,
).add_to(m_colored)

colormap_years.add_to(m_colored)

m_colored.save('stations_map_timelapse_by_year.html')


with open('stations_map_timelapse_by_year.html') as f:
    html = f.read()

html = html.replace('.tickSize(1)', '.tickSize(1)\n        .tickFormat(d3.format("d"))')

with open('stations_map_timelapse_by_year.html', 'w') as f:
    f.write(html)

print("saved stations_map_timelapse_by_year.html")

saved stations_map_timelapse_by_year.html


## Busiest stations and the most popular routes.

In [ ]:
# top-3 busiest stations in the ward

In [36]:
trips_routes = pd.read_parquet(DATA_DIR / 'trips', columns=['start_station_id', 'end_station_id'])

In [41]:
busiest_stations = (
    trips_routes['start_station_id'].value_counts().head(5)
    .rename_axis('station_id').reset_index(name='trips')
)
busiest_stations['name'] = busiest_stations['station_id'].map(stations_full.set_index('station_id')['name'])
print(busiest_stations)


   station_id   trips                                     name
0        7076  376095                  York St / Queens Quay W
1        7006  298031          Bay St / College St (East Side)
2        7016  286991  Bay St / Queens Quay W (Ferry Terminal)
3        7030  262270                  Bay St / Wellesley St W
4        7033  259606                            Union Station


In [39]:
route_pairs = (
    trips_routes[trips_routes['start_station_id'] != trips_routes['end_station_id']]
    .groupby(['start_station_id', 'end_station_id']).size()
    .sort_values(ascending=False).head(5)
)

In [46]:
# stations_full.set_index('station_id')['name']

In [45]:
name_lookup = stations_full.set_index('station_id')['name']
for (s, e), cnt in route_pairs.items():
    print(cnt, name_lookup.get(s, s), '->', name_lookup.get(e, e))

35003 Centre Island Ferry Dock -> Ward's Island Ferry Dock
19550 Ward's Island Ferry Dock -> Centre Island Ferry Dock
19076 Centre Island Ferry Dock -> Centre Island
18233 Centre Island -> Ward's Island Ferry Dock
17522 Centre Island -> Centre Island Ferry Dock


## Users behaviour and seasonal patterns

In [80]:
times = pd.read_parquet(DATA_DIR / 'trips', columns=['trip_id', 'year', 'start_time'], filters = [('year', 'in', [2024, 2025, 2026])])
times.head()

,trip_id,year,start_time
0,26682740,2024,2024-01-01 00:00:08
1,26682742,2024,2024-01-01 00:02:04
2,26682743,2024,2024-01-01 00:03:25
3,26682744,2024,2024-01-01 00:03:39
4,26682746,2024,2024-01-01 00:04:44


In [81]:
times = (times
    .assign(year_month = lambda d: d['start_time'].dt.to_period('M'),
            hour = lambda d: d.start_time.dt.hour)
    )

In [83]:
monthly = (times
    .groupby(['year_month'])['trip_id']
    .size()   
    .reset_index(name='nr_trips')
    )

In [91]:
# 2ad6b1
# FF6B4A
fig_monthly = go.Figure(go.Bar(
    x=monthly['year_month'].dt.to_timestamp(),
    y=monthly['nr_trips'],
    marker=dict(color="#2ad6b1"),
    opacity=0.9
    
))

fig_monthly.update_layout(
    title=dict(text='Trips by Month, 2024-2026', x=0.5),
    xaxis=dict(title='Month', showgrid=False, dtick='M12', tickformat='%Y'),
    yaxis=dict(title='Trips', showgrid=False, tickformat='.2s'),
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1100,
    height=500,
)

fig_monthly

In [80]:
# predict the future. as a time series

## Export: standalone HTML panels (chart + description) for the site

In [326]:
description_html = '''
<p>Ridership peaked in 2025 at 7.8 million trips — <span class="highlight">21,400</span> trips per day on average, <span class="highlight">more than 5x the 2017 rate</span>.</p>
<p>2026 isn't a decline: that year's ridership is still growing, but the data only covers part of the season so far (shown in grey).</p>
'''

chart_div = fig_trips_per_year.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #192562; font-weight: 700; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('trips_per_year_panel.html', 'w') as f:
    f.write(panel_html)

print("saved trips_per_year_panel.html")

saved trips_per_year_panel.html


In [327]:
description_html = '''
<p>The network has grown from 200 stations in 2016 to **1,098** in 2026 — <span class="highlight">more than a 5x expansion</span> in a decade.</p>
<p>The single biggest jump came in 2025, when 169 new stations went live in one year alone.</p>
'''

chart_div = fig_stations_per_year.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .text {{
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
  .chart {{ flex: 3; height: 100%; min-width: 0; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('stations_per_year_panel.html', 'w') as f:
    f.write(panel_html)

print("saved stations_per_year_panel.html")

saved stations_per_year_panel.html


In [328]:
stations_chart_div = fig_stations_per_year.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

tps_chart_div = fig_trips_per_station.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

side_by_side_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .row {{
    display: flex;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .col {{
    width: calc(50% - 16px);
    height: 100%;
    box-sizing: border-box;
  }}
</style>
</head>
<body>
  <div class="row">
    <div class="col">{stations_chart_div}</div>
    <div class="col">{tps_chart_div}</div>
  </div>
</body>
</html>
'''

with open('stations_and_tps_panel.html', 'w') as f:
    f.write(side_by_side_html)

print("saved stations_and_tps_panel.html")

saved stations_and_tps_panel.html


In [329]:
# orange = expansion year (fleet grew 10%+, same threshold as the Stations panel)
# that likely diluted demand; navy = a slower/absorption year where trips per
# station had room to recover. warm/cool rather than red/green, which is one of
# the least colorblind-safe pairings
bar_colors = [
    '#B0B5BA' if y in (2016, 2026) else ('#FF6B4A' if p >= 0.10 else '#1B3A5C')
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]

cycle_stage = [
    'Incomplete year' if y in (2016, 2026) else (
        'Expansion — fleet growing fast' if p >= 0.10 else 'Absorption — demand catching up'
    )
    for y, p in zip(trips_per_station['year'], trips_per_station['yoy_fleet_pct_change'])
]
bar_opacities = [1.0 if y == 2024 else (0.6 if y in (2016, 2026) else 0.45) for y in trips_per_station['year']]

fig_tps_solo = go.Figure(go.Bar(
    x=trips_per_station['year'],
    y=trips_per_station['trips_per_station'],
    marker=dict(color=bar_colors, opacity=bar_opacities),
    customdata=cycle_stage,
    hovertemplate='<b>Year:</b> %{x}<br><b>Value:</b> %{y:,.0f}<br><b>Stage:</b> %{customdata}<extra></extra>',
))

fig_tps_solo.update_layout(
    title=dict(text='Rides Per Station, by Year', x=0.5),
    xaxis=dict(title='Year', tickmode='array', showgrid=False, showline=True, linecolor='black', tickvals=trips_per_station['year']),
    yaxis=dict(title='Trips per station', showgrid=False, showline=True, linecolor='black'),
    showlegend=False,
    hoverlabel=dict(bgcolor='#FFF8E7', bordercolor='#D9A440', font=dict(color='#1a1a1a', size=13)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    autosize=True,
)

description_html = '''
<p>Rides per station peaked at <span class="highlight">7,910</span> in 2024, the year fleet growth slowed the most.</p>
<p>Each expansion wave (2017–2020, then 2025) briefly diluted demand across more stations, pulling the number back down before it recovered.</p>
'''

chart_div = fig_tps_solo.to_html(
    include_plotlyjs='cdn',
    full_html=False,
    default_width='100%',
    default_height='100%',
    config={'responsive': True},
)

panel_html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
  html, body {{ margin: 0; height: 100%; background: #ffffff; }}
  .panel {{
    display: flex;
    align-items: center;
    gap: 32px;
    background: #ffffff;
    padding: 24px;
    box-sizing: border-box;
    height: 100%;
  }}
  .chart {{ order: 1; flex: 3; height: 100%; min-width: 0; }}
  .text {{
    order: 2;
    flex: 2;
    min-width: 0;
    overflow-wrap: break-word;
    color: #1a1a1a;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    font-size: 20px;
    line-height: 1.5;
  }}
  .text p {{ margin: 0 0 20px 0; }}
  .highlight {{ color: #FF6B4A; font-weight: 700; }}
</style>
</head>
<body>
  <div class="panel">
    <div class="text">{description_html}</div>
    <div class="chart">{chart_div}</div>
  </div>
</body>
</html>
'''

with open('trips_per_station_panel.html', 'w') as f:
    f.write(panel_html)

print("saved trips_per_station_panel.html")

saved trips_per_station_panel.html
